# pywswat — Wave Spectral Analysis

| Feature | Class / method |
|---|---|
| Single spectrum (1D / 2D / time-varying) | `Spectra` |
| Multi-item container | `SpectraCollection` |
| Read MIKEIO spectral files | `pywswat.read()` |
| Integrated parameters | `.to_params()` |
| Frequency slice / direction collapse | `.sel_freq()` / `.integrate_dir()` |
| Plotting | `.plot` accessor |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pywswat import Spectra, SpectraCollection, read

DATA = Path("../tests/testdata/spectra")

## 1. Reading MIKEIO spectral files

`pywswat.read()` reads `.dfsu` and `.dfs2` spectral files produced by MIKE models  
(requires `pip install mikeio` or `pip install pywswat[mikeio]`).

### 1a. Point spectrum — `pt_spectra.dfsu`

A single output point from MIKE21SW: shape `(time, freq, dir)`.

In [ ]:
spec_pt = read(DATA / "pt_spectra.dfsu")
spec_pt

In [ ]:
p = spec_pt.to_params()
print(f"Hm0 range : {p['Hm0'].min():.2f} – {p['Hm0'].max():.2f} m")
print(f"Tp  range : {p['Tp'].min():.1f}  – {p['Tp'].max():.1f}  s")
print(f"MWD range : {p['MWD'].min():.0f}  – {p['MWD'].max():.0f}  °")

In [ ]:
spec_pt.plot.timeseries(params=["Hm0", "Tp", "MWD"], figsize=(8, 5))
plt.tight_layout()
plt.show()

In [ ]:
# Plot spectrum at first time step
spec_pt.plot.spectrum(figsize=(7, 4), title="Wave spectrum — t=0")
plt.tight_layout()
plt.show()

### 1b. Area spectrum — `area_spectra.dfsu`

40 mesh elements × 3 time steps.  Without `location`, all elements are returned  
as a `SpectraCollection`.

In [ ]:
col_area = read(DATA / "area_spectra.dfsu")
col_area

In [ ]:
# Integrated parameters for all elements — outer key = element name
params_all = col_area.to_params()
hm0_all = np.array([p["Hm0"] for p in params_all.values()])  # (n_elements, nt)
print(f"Hm0 across all elements and timesteps: {hm0_all.min():.2f} – {hm0_all.max():.2f} m")

In [ ]:
# Pick a single element and inspect its spectrum
spec_elem = read(DATA / "area_spectra.dfsu", location=5)
print(spec_elem)
spec_elem.plot.spectrum(figsize=(7, 4), title="Element 5 — t=0")
plt.tight_layout()
plt.show()

### 1c. Line spectrum — `line_spectra.dfsu`

Spectra along a transect: 10 nodes × 4 time steps.

In [ ]:
col_line = read(DATA / "line_spectra.dfsu")
print(col_line)

# Compare Hm0 at first and last node
params_line = col_line.to_params()
for name in ["element_0", "element_9"]:
    hm0 = params_line[name]["Hm0"]
    print(f"  {name}: Hm0 = {hm0}")

### 1d. DFS2 spectral file — `dir_wave_analysis_spectra.dfs2`

Single time step.  The grid x/y directly encode frequency (Hz) and direction (°).

In [ ]:
spec_dfs2 = read(DATA / "dir_wave_analysis_spectra.dfs2")
print(spec_dfs2)
spec_dfs2.plot.spectrum(figsize=(7, 4), title="DFS2 spectrum")
plt.tight_layout()
plt.show()

### 1e. Swell / wind-sea partition from a real file

In [ ]:
spec = read(DATA / "pt_spectra.dfsu")

swell    = spec.sel_freq(fmin=0.04, fmax=0.10)
wind_sea = spec.sel_freq(fmin=0.10, fmax=0.50)

col = SpectraCollection({"swell": swell, "wind_sea": wind_sea, "total": spec})
col_params = col.to_params()

print(f"{'Component':<10} {'Hm0 mean (m)':>14} {'Tp mean (s)':>12}")
print("-" * 38)
for name, p in col_params.items():
    print(f"{name:<10} {p['Hm0'].mean():>14.3f} {p['Tp'].mean():>12.2f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, (name, s) in zip(axes, col._items.items()):
    s.plot.timeseries(params=["Hm0"], ax=ax)
    ax.set_title(name)
    ax.set_xlabel("")
axes[-1].set_xlabel("Time step")
plt.tight_layout()
plt.show()

---

## 2. Frequency-only spectrum (synthetic)

A narrow Gaussian centred at `fp = 0.1 Hz` (`Tp ≈ 10 s`).

In [ ]:
freq  = np.linspace(0.03, 0.5, 128)
fp    = 0.1
sigma = 0.015
energy = np.exp(-0.5 * ((freq - fp) / sigma) ** 2)

spec_1d = Spectra(energy, freq=freq)
spec_1d

In [ ]:
p = spec_1d.to_params()
print(f"Hm0  = {p['Hm0']:.3f} m")
print(f"Tp   = {p['Tp']:.2f} s  (expected ~{1/fp:.1f} s)")
print(f"T01  = {p['T01']:.2f} s")
print(f"T02  = {p['T02']:.2f} s")
print(f"Tm10 = {p['Tm10']:.2f} s")

## 3. 2-D spectrum (frequency × direction)

In [ ]:
dirs     = np.arange(0, 360, 10, dtype=float)
mean_dir = 225.0

D = np.maximum(0.0, np.cos(np.deg2rad(dirs - mean_dir)) ** 2)
D /= D.sum()

energy_2d = np.outer(energy, D)
spec_2d = Spectra(energy_2d, freq=freq, direction=dirs)

p2 = spec_2d.to_params()
print(f"Hm0={p2['Hm0']:.3f} m  Tp={p2['Tp']:.2f} s  MWD={p2['MWD']:.1f}°  DSD={p2['DSD']:.3f} rad")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
spec_2d.plot.spectrum(ax=axes[0], title="2-D spectrum")
spec_1d.plot.spectrum(ax=axes[1], title="1-D frequency spectrum")
plt.tight_layout()
plt.show()

## 4. Time-varying spectrum

In [ ]:
nt        = 24
time      = np.arange(nt, dtype=float)
fp_series = np.linspace(0.08, 0.15, nt)

energy_t = np.stack(
    [np.exp(-0.5 * ((freq - fp_t) / sigma) ** 2) for fp_t in fp_series]
)

spec_t = Spectra(energy_t, freq=freq, time=time)
spec_t.plot.timeseries(params=["Hm0", "Tp"], figsize=(7, 4))
plt.tight_layout()
plt.show()

## 5. SpectraCollection (synthetic)

In [ ]:
e_swell = np.exp(-0.5 * ((freq - 0.08) / 0.008) ** 2)
e_wind  = np.exp(-0.5 * ((freq - 0.20) / 0.030) ** 2) * 0.4

col = SpectraCollection({
    "swell" : Spectra(e_swell,            freq=freq),
    "wind"  : Spectra(e_wind,             freq=freq),
    "total" : Spectra(e_swell + e_wind,   freq=freq),
})

all_params = col.to_params()
print(f"{'Item':<8} {'Hm0 (m)':>10} {'Tp (s)':>10}")
print("-" * 30)
for name, p in all_params.items():
    print(f"{name:<8} {p['Hm0']:>10.3f} {p['Tp']:>10.2f}")

## 6. Direction-only spectrum

In [ ]:
dir_energy = np.maximum(0.0, np.cos(np.deg2rad(dirs - 270.0)) ** 4)
spec_dir = Spectra(dir_energy, direction=dirs)
print(spec_dir)
print("to_params() →", spec_dir.to_params())  # empty — no freq axis